In [4]:
from pathlib import Path
import pandas as pd

DATADIR = '/mnt/d/WSI/CAMELYON16'
assert Path(DATADIR).exists()

df = pd.read_csv(f'{DATADIR}/csv/labeled_camelyon16.csv')

In [13]:
'''
LABELS
0 -> normal
1 -> abnormal
2 -> normal patch in abnormal slide
3 -> patches in the border of the annotation
'''

positive_df = df.query("slide_label == 1")
pd.pivot_table(positive_df, index=['subset', 'label'], values=['img_path'], aggfunc='count')

img_path
subset label          
test   1         54129
       2        210564
       3          4550
train  1         64924
       2        600610
       3         10256

In [11]:
df

,subset,slide,img_path,img_size_kb,label,slide_label
0,train,normal_001,/mnt/d/WSI/CAMELYON16/camelyon16_patch_256_20x...,25.131836,0,0
1,train,normal_001,/mnt/d/WSI/CAMELYON16/camelyon16_patch_256_20x...,22.099609,0,0
2,train,normal_001,/mnt/d/WSI/CAMELYON16/camelyon16_patch_256_20x...,22.414062,0,0
3,train,normal_001,/mnt/d/WSI/CAMELYON16/camelyon16_patch_256_20x...,22.772461,0,0
4,train,normal_001,/mnt/d/WSI/CAMELYON16/camelyon16_patch_256_20x...,18.392578,0,0
...,...,...,...,...,...,...
2271992,test,test_130,/mnt/d/WSI/CAMELYON16/camelyon16_patch_256_20x...,22.602539,0,0
2271993,test,test_130,/mnt/d/WSI/CAMELYON16/camelyon16_patch_256_20x...,23.920898,0,0
2271994,test,test_130,/mnt/d/WSI/CAMELYON16/camelyon16_patch_256_20x...,20.109375,0,0
2271995,test,test_130,/mnt/d/WSI/CAMELYON16/camelyon16_patch_256_20x...,16.099609,0,0


In [22]:
# DIVIDING TRAIN/VAL BY SLIDES
import random

train_df = df.query("subset=='train'")
new_df = df.query("subset=='test'")

train_perc = .8

for label in [0, 1]:
    random.seed(2024)
    subdf = train_df.query(f"slide_label=={label}")
    slide_names = subdf['slide'].unique()
    n_train = int(len(slide_names) * train_perc)
    selected_slides = random.sample(list(slide_names), n_train)

    train_subdf = subdf.query(f"slide in {selected_slides}")
    val_subdf = subdf.query(f"slide not in {selected_slides}")
    val_subdf['subset'] = 'val'

    new_df = pd.concat([new_df, train_subdf, val_subdf], axis=0)


/tmp/ipykernel_948594/2092006442.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  val_subdf['subset'] = 'val'
/tmp/ipykernel_948594/2092006442.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  val_subdf['subset'] = 'val'


In [30]:
import numpy as np

pd.pivot_table(new_df, index=['subset', 'slide_label'], values=['slide'], aggfunc=lambda x: np.unique(x).shape[0])

slide
subset slide_label       
test   0               88
       1               41
train  0              135
       1               80
val    0               34
       1               21

In [35]:
train_slides = new_df.query("subset=='train'")['slide'].unique()
val_slides = new_df.query("subset=='val'")['slide'].unique()

set(train_slides) & set(val_slides)

set()

In [36]:
new_df.to_csv("csv/camelyon16_data.csv", index=False)

In [37]:
'''
LABELS
0 -> normal
1 -> abnormal
2 -> normal patch in abnormal slide
3 -> patches in the border of the annotation
'''


,subset,slide,img_path,img_size_kb,label,slide_label
1542569,test,test_001,/mnt/d/WSI/CAMELYON16/camelyon16_patch_256_20x...,17.959961,2,1
1542570,test,test_001,/mnt/d/WSI/CAMELYON16/camelyon16_patch_256_20x...,17.202148,2,1
1542571,test,test_001,/mnt/d/WSI/CAMELYON16/camelyon16_patch_256_20x...,17.750977,2,1
1542572,test,test_001,/mnt/d/WSI/CAMELYON16/camelyon16_patch_256_20x...,16.882812,2,1
1542573,test,test_001,/mnt/d/WSI/CAMELYON16/camelyon16_patch_256_20x...,17.947266,2,1
...,...,...,...,...,...,...
1542564,val,tumor_111,/mnt/d/WSI/CAMELYON16/camelyon16_patch_256_20x...,20.204102,2,1
1542565,val,tumor_111,/mnt/d/WSI/CAMELYON16/camelyon16_patch_256_20x...,20.218750,2,1
1542566,val,tumor_111,/mnt/d/WSI/CAMELYON16/camelyon16_patch_256_20x...,21.076172,2,1
1542567,val,tumor_111,/mnt/d/WSI/CAMELYON16/camelyon16_patch_256_20x...,19.930664,2,1


In [38]:
import torch

a = torch.load("models/supervised_resnet18.pth")

In [40]:
a['state_dict']

OrderedDict([('backbone.conv1.weight',
              tensor([[[[ 2.5167e-02,  5.4288e-02, -3.5951e-02,  ...,  7.7984e-02,
                          1.3417e-02, -9.3965e-03],
                        [-2.0568e-02,  5.3696e-02, -5.8493e-02,  ...,  7.4035e-02,
                         -4.0631e-02,  1.9521e-03],
                        [-2.4114e-02,  1.5787e-01, -7.6331e-02,  ...,  8.4116e-02,
                         -4.4171e-02,  3.5973e-02],
                        ...,
                        [ 4.5541e-02,  1.6133e-01, -3.6068e-01,  ...,  7.0760e-02,
                         -1.7587e-02,  1.9499e-02],
                        [ 6.6744e-02,  1.4916e-02, -2.2822e-01,  ...,  6.9160e-03,
                          3.3283e-02,  1.1854e-02],
                        [ 2.8907e-02, -2.5122e-03, -1.1278e-01,  ..., -2.7575e-02,
                          3.6057e-02, -2.1331e-02]],
              
                       [[-4.4951e-02,  1.0936e-01,  5.3916e-02,  ..., -7.6071e-03,
                       